# Extraire le texte dans des documents "riches"

On utilise ici `markitdown` (Microsoft) comme librairie principale d'extraction, avec `docx2txt`, `striprtf`, `odfpy`, `easyocr` et `openai-whisper` en secours pour les formats qu'elle ne couvre pas ou mal. Ce sont des librairies Python pures, installables via `uv sync` / `pip install`, sans programme externe à configurer sur Windows — à une exception près :

- **ffmpeg** est nécessaire pour que `openai-whisper` puisse lire les fichiers audio (mp3, wav, ...). Sous Windows, téléchargez un build depuis [gyan.dev](https://www.gyan.dev/ffmpeg/builds/), décompressez l'archive et ajoutez le dossier `bin` à la variable d'environnement `Path`. Redémarrez le terminal ensuite.

Limite restante par rapport à `textract` : les anciens fichiers `.doc` (binaires, Word 97-2003) ne sont pas supportés — voir la note plus bas.

## Imports

In [ ]:
import os
import docx2txt
import easyocr
import openpyxl
import whisper
from markitdown import MarkItDown
from odf import teletype, text as odf_text
from odf.opendocument import load as load_odt
from striprtf.striprtf import rtf_to_text

## Lister les fichiers dans le répertoire `dummy`

In [ ]:
path = 'dummy/'
files = os.listdir(path)
print(files)

## Extraire le texte de chacun des fichiers

`markitdown` gère directement la plupart des formats (`.pdf`, `.docx`, `.pptx`, `.xls`, `.xlsx`, `.txt`, ...). On ajoute des solutions de secours (*fallback*) pour les formats qu'il ne couvre pas ou mal :
- `docx2txt`, utilisé si `markitdown` ne parvient pas à extraire de texte d'un `.docx`
- `openai-whisper`, un modèle de transcription audio, utilisé pour les fichiers `.mp3`
- `striprtf`, pour nettoyer le texte brut d'un `.rtf` (markitdown n'a pas de convertisseur dédié pour ce format et renvoie les balises RTF telles quelles)
- `odfpy`, pour lire le texte d'un `.odt` (OpenDocument, le format de LibreOffice/OpenOffice)
- `easyocr`, un modèle d'OCR (reconnaissance de texte dans une image), utilisé pour les fichiers `.jpg`/`.png`. Contrairement à `pytesseract`, il ne nécessite pas d'installer le binaire `tesseract`

Les anciens fichiers `.doc` (format binaire Word 97-2003) restent hors du champ de ce prototype : il n'existe pas de librairie Python pure pour les lire, seulement des solutions qui nécessitent un programme externe (LibreOffice, Word...). Le plus simple est de les réenregistrer au format `.docx` avant de les traiter.
</cell id="cell-6">

In [ ]:
md_converter = MarkItDown()
# "tiny" est le plus petit modèle whisper : rapide, mais moins précis que "base" ou "small"
whisper_model = whisper.load_model("tiny")
ocr_reader = easyocr.Reader(["fr", "en"], gpu=False)


def extract_text(filepath):
    ext = os.path.splitext(filepath)[1].lower()

    if ext == ".mp3":
        return whisper_model.transcribe(filepath)["text"].strip()

    if ext in (".jpg", ".jpeg", ".png"):
        return " ".join(ocr_reader.readtext(filepath, detail=0)).strip()

    if ext == ".rtf":
        with open(filepath, encoding="utf-8", errors="ignore") as f:
            return rtf_to_text(f.read()).strip()

    if ext == ".odt":
        doc = load_odt(filepath)
        paragraphs = doc.getElementsByType(odf_text.P)
        return " ".join(teletype.extractText(p) for p in paragraphs).strip()

    try:
        text = md_converter.convert(filepath).text_content.strip()
        if text:
            return text
    except Exception:
        pass

    if ext == ".docx":
        return docx2txt.process(filepath).strip()

    raise ValueError(f"Aucun convertisseur n'a réussi à extraire le texte de {filepath}")


for f in sorted(files):
    filepath = os.path.join(path, f)
    try:
        print(f, "->", extract_text(filepath))
    except Exception as e:
        print(filepath, e)